# Fuselage Design
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

Size the **fuselage** of the tail-sitter from the converged mass closure and the
EDF geometry:

- internal **packaging** of battery, payload, and avionics,
- maximum **diameter** and **length** (fineness ratio trade),
- **wetted area** and the fuselage contribution to $C_{D0}$,
- **shell mass** vs. the structural weight-fraction budget,
- **longitudinal layout, CG**, and wing placement for static margin,
- cross-check of the **control-vane moment arm** assumed in Notebook 3.

---

## Axis Convention  (Aetherion-compatible)

All geometry follows the **Aetherion** 6-DoF library frames:

| Frame | Convention |
|-------|-----------|
| **World** | NED — $x$ North, $y$ East, $z$ Down |
| **Body**  | FRD — $x$ **forward** (out the nose), $y$ **right**, $z$ **down** |

Fuselage **stations** $x_s$ are measured from the **nose tip, positive aft**
(standard practice).  Conversion to the body frame is:

$$ x_{body} = -x_s $$

The EDF exhaust and the control vanes point in $-x_{body}$.
In hover the body $+x$ axis points up (tail-sitter attitude).

---

## Inputs

- `config/*.yaml` — sizing loop re-run from configuration
- `out/airfoil.yaml` — wing airfoil
- `out/control_vanes.yaml` — EDF hub radius, vane geometry, servo mass
- `out/aileron.yaml` — aileron servo/linkage mass, carved from the
  avionics/structural fractions like the vane hardware
- `out/vibration.yaml` — isolator hardware mass (carved) and the sway/rattle
  space fed into the bay stack
- `config/fuselage.yaml` — packaging densities, shape and shell parameters

## Outputs

- `out/fuselage.yaml`
- `out/fuselage_layout.png`
- `out/fuselage_drag_trade.png`

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import numpy as np

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design import reports
from conceptual_design.plots import plot_fineness_trade, plot_fuselage_layout

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` — same pattern as NB2 and NB3, so this
notebook stays consistent with the upstream design state.  Nothing is
hard-coded; all numbers come from configuration and the previous notebooks.

---

In [ ]:
# -- Re-run the sizing loop + read every upstream handoff ----------------------
inp, result = solve_design_point(CONFIG_PATH)

af    = load_handoff(OUT_PATH, "airfoil")
vanes = load_handoff(OUT_PATH, "control_vanes")
ail   = load_handoff(OUT_PATH, "aileron")
vib   = load_handoff(OUT_PATH, "vibration")

MTOW_kg      = result.m_total_kg
chord_mean_m = result.wing.chord_mean
S_wing       = result.wing.S_wing
R_hub_m      = vanes["R_hub_m"]
m_aileron_servo_kg   = ail["servo_mass_kg_each"]   * ail["n_ailerons"]
m_aileron_linkage_kg = ail["linkage_mass_kg_each"] * ail["n_ailerons"]
m_isolation_avionics_kg = vib["m_isolation_avionics_kg"]
m_isolation_struct_kg   = vib["m_isolation_struct_kg"]
sway_pad_m              = vib["sway_pad_total_m"]

print(f"Converged MTOW : {MTOW_kg:.3f} kg")
print(f"Battery        : {result.m_battery_kg:.3f} kg")
print(f"Payload        : {result.m_payload_kg:.3f} kg")
print(f"Avionics       : {result.m_avionics_kg:.3f} kg")
print(f"D_rotor        : {inp.rotor.D_rotor_m*1e3:.0f} mm   EDF hub radius : {R_hub_m*1e3:.1f} mm")
print(f"Wing           : S = {S_wing:.4f} m^2,  b = {result.wing.b_wing:.3f} m,  MAC = {chord_mean_m*1e3:.1f} mm")
print(f"Wing airfoil   : {af['designation']}")
print(f"Aileron hw     : {(m_aileron_servo_kg + m_aileron_linkage_kg)*1e3:.1f} g "
      f"(NB4 -- {ail['n_ailerons']} x servo + linkage)")
print(f"Isolation hw   : {(m_isolation_avionics_kg + m_isolation_struct_kg)*1e3:.1f} g, "
      f"sway pad {sway_pad_m*1e3:.2f} mm (NB5 -- f_n {vib['f_n_hz']:.0f} Hz, "
      f"{100*(1-vib['target_transmissibility']):.0f}% 1/rev cut)")

# Section 2 — Packaging Volume Budget

Each internal component occupies a cylindrical **bay** of the mid-body:

$$
V_{comp} = \frac{m_{comp}}{\rho_{comp}}, \qquad
L_{bay} = \frac{V_{comp}}{\eta_{pack} \, A_{int}}, \qquad
A_{int} = \pi \left(\tfrac{D_{fus}}{2} - t_{shell}\right)^2
$$

The packing factor $\eta_{pack}$ accounts for corner losses against the curved
shell, connectors, foam, and access clearances.  The bay stack (payload →
avionics → battery) must fit between the nose equipment station and the start
of the tail cone:

$$
\sum L_{bay} \;\le\; L_{mid} + \tfrac{1}{2} L_{nose}
$$

---

In [ ]:
from conceptual_design.fuselage_design import (
    FuselageParams, ModularityParams, component_volumes,
)

fus_p = FuselageParams.from_yaml(CONFIG_PATH / "fuselage.yaml")
mod_p = ModularityParams.from_yaml(CONFIG_PATH / "modularity.yaml")   # ADR-0008 split lines
vols  = component_volumes(result.m_battery_kg, result.m_payload_kg,
                          result.m_avionics_kg, fus_p)

print(f"{'component':<10} {'mass [kg]':>10} {'rho [kg/m3]':>12} {'volume [L]':>11}")
rho_map = {"payload": fus_p.rho_payload, "avionics": fus_p.rho_avionics,
           "battery": fus_p.rho_battery_pack}
mass_map = {"payload": result.m_payload_kg, "avionics": result.m_avionics_kg,
            "battery": result.m_battery_kg}
for k, v in vols.items():
    print(f"{k:<10} {mass_map[k]:>10.3f} {rho_map[k]:>12.0f} {v*1e3:>11.3f}")
V_bays = sum(vols.values())
print("-" * 46)
print(f"{'total':<10} {sum(mass_map.values()):>10.3f} {'':>12} {V_bays*1e3:>11.3f}")
print(f"\nWith packing factor {fus_p.packing_factor:.2f} -> "
      f"{V_bays/fus_p.packing_factor*1e3:.2f} L of bay volume required")

# Section 3 — Diameter and Length

Two constraints compete for the maximum diameter:

**1. Hub constraint (geometric).**  On a tail-sitter EDF the tail cone *is* the
fan centerbody — it must taper **down** to the hub, so:

$$
D_{fus} \;\ge\; k_{hub} \cdot 2 R_{hub}
$$

**2. Packaging constraint.**  At the target fineness ratio $f = L/D$, the bay
stack must fit in the available length (Section 2).

The module takes the **smallest diameter satisfying both** and reports which
constraint is active.  The body of revolution has three segments:

| Segment | Shape | Length |
|---------|-------|--------|
| Nose | semi-ellipsoid | $f_{nose} \, L$ |
| Mid  | cylinder | $(1 - f_{nose} - f_{tail})\, L$ |
| Tail | smoothstep boattail to $R_{hub}$ | $f_{tail}\, L$ |

The boattail uses $r(s) = R - (R - R_{hub})\, s^2 (3 - 2s)$ — tangent to the
cylinder at entry, level at the hub — and the same function later drives the
CAD revolve profile, so the geometry is defined in exactly one place.

---

In [ ]:
from conceptual_design.fuselage_design import size_fuselage

# The full size_fuselage input set, built once -- the fineness sweep below
# re-uses it with only the fineness ratio replaced.
fus_kwargs = dict(
    m_battery_kg    = result.m_battery_kg,
    m_payload_kg    = result.m_payload_kg,
    m_avionics_kg   = result.m_avionics_kg,
    m_propulsion_kg = result.m_propulsion_kg,
    m_structure_kg  = result.m_structure_kg,
    m_wing_kg       = result.m_wing_kg,
    m_misc_kg       = result.m_misc_kg,
    R_hub_m         = R_hub_m,
    D_rotor_m       = inp.rotor.D_rotor_m,
    c_vane_m        = vanes["c_vane_m"],
    n_vanes         = vanes["n_vanes"],
    S_vane_m2       = vanes["S_vane_m2"],
    hinge_xc        = vanes["hinge_xc"],
    chord_mean_m    = chord_mean_m,
    m_aileron_servo_kg   = m_aileron_servo_kg,
    m_aileron_linkage_kg = m_aileron_linkage_kg,
    m_isolation_avionics_kg = m_isolation_avionics_kg,
    m_isolation_struct_kg   = m_isolation_struct_kg,
    sway_pad_m              = sway_pad_m,
    V_cruise        = inp.mission.V_cruise,
    rho             = inp.env.rho,
    S_wing          = S_wing,
    mod             = mod_p,
    construction_method = inp.wf.construction_method,
)

fus = size_fuselage(**fus_kwargs, p=fus_p)
reports.print_fuselage_sizing(fus)

# Section 4 — Fineness Ratio Trade

With the diameter pinned by the hub, fineness ratio trades **wetted area /
shell mass** against **form drag**.  Raymer's component buildup:

$$
Re_L = \frac{\rho V L}{\mu}, \qquad
C_f = \frac{0.455}{(\log_{10} Re_L)^{2.58}}, \qquad
FF = 1 + \frac{60}{f^3} + \frac{f}{400}
$$

$$
C_{D0,fus} = C_f \cdot FF \cdot Q \cdot \frac{S_{wet}}{S_{wing}}
$$

Short bodies pay in form factor ($60/f^3$), long bodies pay in skin friction
area.  The curve is flat near its minimum — the choice is dominated by
packaging and by keeping the tail-sitter compact enough to stand on its duct.

---

In [ ]:
from conceptual_design.fuselage_design import fineness_sweep

FR_sweep = np.linspace(3.0, 8.0, 40)
cd0_sweep, mshell_sweep = fineness_sweep(fus_kwargs, fus_p, FR_sweep)

plot_fineness_trade(FR_sweep, cd0_sweep, mshell_sweep, fus, fus_p,
                    OUT_PATH / "fuselage_drag_trade.png")

# Section 5 — Drag Budget Check

The conceptual sizing (NB1) assumed a total $C_{D0}$ in
`config/aerodynamics.yaml`.  Now that the two dominant components are actually
designed, verify the assumption still closes:

- wing profile drag from the NB2 airfoil analysis ($C_{d0}$ section, scaled ~1.10
  for the exposed wetted surface ratio),
- fuselage from this notebook's buildup,
- small allowances for duct external surface, vanes, and interference.

---

In [ ]:
CD0_misc = 0.0025    # duct external + vanes + antennas + interference
reports.print_drag_budget(fus, af, inp.aero, S_wing, CD0_misc)

# Section 6 — Shell Mass Check

Equivalent-monocoque area-density model:

$$
m_{shell} = k_{struct} \cdot \rho_{shell} \cdot t_{shell} \cdot S_{wet}
$$

compared against the structural weight-fraction budget from the mass closure,
**minus** the wing (sized separately by the Raymer formula in NB1):

$$
m_{budget} = m_{structure} - m_{wing}
$$

The duct ring is *not* charged here — shrouds are booked in the propulsion
fraction.

---

In [ ]:
print(f"Shell + frames mass (model)   : {fus.m_shell_kg:.3f} kg")
print(f"Structural budget (fraction)  : {fus.m_struct_budget_kg:.3f} kg")
util = fus.m_shell_kg / fus.m_struct_budget_kg * 100
print(f"Budget utilisation            : {util:.0f}%")
if fus.m_shell_kg <= fus.m_struct_budget_kg:
    print("Shell mass CLOSES within the structural fraction.")
else:
    print("Shell mass EXCEEDS budget -- increase fs or thin the laminate.")

# Section 7 — Longitudinal Layout and CG

Components stack nose-to-tail: **payload → avionics → battery → ESC**, with the
motor + fan at the fan plane in the boattail and the duct ring at the tail.
The propulsion fraction is split 60 / 25 / 15 % between motor+fan / ESC / duct.

The **wing is placed last**: its aerodynamic center goes $SM \cdot MAC$ behind
the all-up CG so the cruise static margin is met by construction:

$$
x_{AC} = x_{CG} + SM \cdot \bar{c}, \qquad x_{LE} = x_{AC} - 0.25\,\bar{c}
$$

Stations from the nose, positive aft;  $x_{body} = -x_s$ (Aetherion FRD).

---

In [ ]:
plot_fuselage_layout(fus, fus_p, vanes, chord_mean_m,
                     "Fuselage internal layout — side view (body $-z$ up in plot)",
                     OUT_PATH / "fuselage_layout.png")

reports.print_layout_table(fus, MTOW_kg, fus_p)

# Section 8 — Control Moment Arm Cross-Check

Notebook 3 sized the jet vanes with an **assumed** CG-to-vane arm
$L_{CG}$.  The layout above gives the *actual* arm.  Since the achievable
control moment is $M = F_{vane} \, L$, the ratio of actual to assumed arm is a
direct **control authority margin** on the NB3 result.

---

In [ ]:
reports.print_vane_arm_check(fus, vanes)

# Section 9 — Output Export

## Output File

$$
\texttt{out/fuselage.yaml}
$$

Handoff interface for downstream use:

- **CAD notebook** — revolve profile parameters, duct geometry, wing/vane
  stations *(all stations from nose, +aft; $x_{body} = -x_s$)*
- **Flight-dynamics setup (Aetherion)** — CG station, static margin, layout
  table for inertia estimation

---

In [ ]:
from conceptual_design.fuselage_design import write_fuselage_yaml

write_fuselage_yaml(fus, fus_p, OUT_PATH / "fuselage.yaml")
print(f"Fuselage design written -> {OUT_PATH / 'fuselage.yaml'}")

# Section 10 — Design Summary

In [ ]:
reports.print_fuselage_summary(fus, fus_p, vanes)